# OpenAI — Explanations vs Built-in Reasoning

Notebook version of `openai/openai_cot_thinking.py`.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

PROBLEM = "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?"

## 1. Requested step-by-step explanation

This does **not** enable a special hidden reasoning mode. It asks the model to include an explanation in the visible answer; that generated explanation is not access to raw internal reasoning.

In [ ]:
explanation_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            # This prompt requests a visible step-by-step explanation.
            # It does not expose the model's raw internal reasoning.
            "content": "Think step by step before giving your final answer.",
        },
        {"role": "user", "content": PROBLEM},
    ],
)
print(explanation_response.choices[0].message.content)

## 2. Built-in reasoning via the Responses API

`gpt-5.6` reasons internally before answering. `reasoning.effort` controls how much reasoning it performs; supported values depend on the selected model.

In [ ]:
thinking_response = client.responses.create(
    model="gpt-5.6",
    reasoning={"effort": "medium"},
    input=PROBLEM,
)

# output_text contains the final answer, not the model's raw internal reasoning.
# Supported models can optionally return a reasoning summary.
print("Answer:", thinking_response.output_text)

In [ ]:
# Token usage:
#   input     = tokens in the input you sent
#   reasoning = hidden internal thinking tokens used by the reasoning model
#   output    = all output-side tokens, including hidden reasoning tokens
if (usage := thinking_response.usage) and usage.output_tokens_details:
    print(f"Tokens — input: {usage.input_tokens}, "
          f"reasoning: {usage.output_tokens_details.reasoning_tokens}, "
          f"output: {usage.output_tokens}")

## 3. Optional reasoning summary

A reasoning summary is generated by the model; it is not the raw internal reasoning. Running the next cell makes an additional API request.

In [ ]:
summary_response = client.responses.create(
    model="gpt-5.6",
    reasoning={"effort": "medium", "summary": "auto"},
    input=PROBLEM,
)

for item in summary_response.output:
    if item.type == "reasoning":
        for summary_part in item.summary:
            print(summary_part.text)
print("Answer:", summary_response.output_text)